# **PYSPARK**

### **Customers**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema" ,True)\
        .load("/Volumes/databricksansh/bronze/bronze_volume/customers/")

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df  = df.withColumn("name", upper(col("name")))

In [0]:
display(df)

In [0]:
df = df.withColumn("domain",split(col("email"),"@")[1])
display(df)

In [0]:
display(df.groupBy("domain").agg(count(col("customer_id")).alias("Total")).sort(col("Total").desc()))


Databricks visualization. Run in Databricks to view.

In [0]:
df = df.withColumn("processDate" , current_timestamp())
display(df)

### **UPSIRT**

In [0]:
from delta.tables import DeltaTable

### **customers**

In [0]:
if spark.catalog.tableExists("databricksansh.silver.customers_enr"):
    # this is first step suppose you want to merge so first you have object 
  dlt_obj = DeltaTable.forName(spark , "databricksansh.silver.customers_enr")
  dlt_obj.alias("trg").merge(df.alias("src"), "trg.customer_id = src.customer_id")\
      .whenMatchedUpdateAll()\
      .whenNotMatchedInsertAll()\
      .execute()
else:
  df.write.format("delta")\
      .mode("append")\
      .saveAsTable("databricksansh.silver.customers_enr") 

In [0]:
%sql
select * from databricksansh.silver.customers_enr;

### **Products**

In [0]:
df_prod = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/databricksansh/bronze/bronze_volume/products/dim_products.csv")
display(df_prod)

In [0]:
df_prod.printSchema()

In [0]:
df_prod= df_prod.withColumn("processDate",current_timestamp())
display(df_prod)


In [0]:
display(df_prod.groupBy("category").agg(avg("price").alias("avg_price"))\
    .sort(col("avg_price").desc()))

Databricks visualization. Run in Databricks to view.

In [0]:
if spark.catalog.tableExists("databricksansh.silver.products_enr"):
    dlt_obj = DeltaTable.forName(spark , "databricksansh.silver.products_enr")
    dlt_obj.alias("trg").merge(df_prod.alias("src"), "trg.product_id = src.product_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_prod.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksansh.silver.products_enr")

In [0]:
%sql
select * from databricksansh.silver.products_enr

### **STORES**

In [0]:
df_str = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/databricksansh/bronze/bronze_volume/stores/")
display(df_str)

In [0]:
df_str = df_str.withColumn("store_name",regexp_replace(col("store_name"),"_" , ""))
display(df_str)

In [0]:
df_str = df_str.withColumn("processDate" , current_timestamp())
display(df_str)

In [0]:
if spark.catalog.tableExists("databricksansh.silver.stores_enr"):
    dlt_obj = DeltaTable.forName(spark , "databricksansh.silver.stores_enr")
    dlt_obj.alias("trg").merge(df_str.alias("src"),"trg.store_id = src.store_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_str.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksansh.silver.stores_enr")

In [0]:
%sql
select * from databricksansh.silver.stores_enr

### **SALES**

In [0]:
df_sales = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema", True)\
    .load("/Volumes/databricksansh/bronze/bronze_volume/sales/")
display(df_sales)

In [0]:
df_sales = df_sales.withColumn("pricePerSale", round(col("total_amount")/col("quantity"),2))
df_sales = df_sales.withColumn("processDate",current_timestamp())
display(df_sales)

In [0]:
if spark.catalog.tableExists("databricksansh.silver.sales_enr"):
    dlt_obj = DeltaTable.forName(spark , "databricksansh.silver.sales_enr" )
    dlt_obj.alias("trg").merge(df_sales.alias("src"), "trg.sales_id = src.sales_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_sales.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksansh.silver.sales_enr")

In [0]:
%sql
select * from databricksansh.silver.sales_enr

# **Spark SQL**

In [0]:
display(spark.sql("select * from databricksansh.silver.sales_enr"))

In [0]:
df = (spark.sql("select UPPER(category) from databricksansh.silver.products_enr"))
display(df)

In [0]:
df.createOrReplaceTempView("temp_products")

In [0]:
df = spark.sql("""
          SELECT * , 
            CASE
            WHEN `UPPER(category)` = 'TOYS' THEN 'Yes' ELSE 'NO' END AS flag
            FROM temp_products
          """)

In [0]:
display(df)

# **PYSPARK UDF **

In [0]:
def greet(i):
    return "hello" + str(i)

In [0]:
udf_greet = udf(greet)

In [0]:
df = df.withColumn("greet",udf_greet(col("flag")))
display(df)